# Phase 1: Pretraining HybridTRM on FineWeb-Edu

This notebook handles pretraining the recursive reasoning model on the FineWeb-Edu dataset.

## Training Goals
- **Dataset**: FineWeb-Edu (sample-10BT)
- **Target Tokens**: 10 Billion
- **Model**: GPT-2 Small (124M) with 64 latent tokens
- **Hardware**: 2x A40 GPUs (48GB each)
- **Duration**: ~1-2 days

## Key Features
- Streaming dataset (no local storage needed)
- Deep supervision with latent refinement loops
- Automatic checkpointing every 5000 steps
- Evaluation every 2000 steps
- Weights & Biases logging

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
from transformers import GPT2Tokenizer
from accelerate import Accelerator

from src.config import ModelConfig, PretrainingConfig, WandbConfig
from src.model import HybridTRM
from src.data import get_pretrain_dataloader, create_validation_dataset
from src.training import train, setup_scheduler
from src.utils import (
    enable_tf32,
    setup_wandb,
    load_checkpoint,
    get_latest_checkpoint,
    print_model_info,
    finish_wandb,
)

print("✓ Imports successful")

## Configuration

In [ ]:
# Model configuration
model_config = ModelConfig(
    base_model="gpt2",  # GPT-2 Small (124M)
    n_latents=64,
    n_sup=8,
    t_loops=3,
    seq_len_x=512,
    seq_len_y=512,
    use_rope=True,  # Use RoPE instead of learned positional embeddings
    gradient_checkpointing=True,  # Enable for memory efficiency
)

# Pretraining configuration
train_config = PretrainingConfig(
    dataset_name="HuggingFaceFW/fineweb-edu",
    dataset_subset="sample-10BT",
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=2000,
    target_tokens=10_000_000_000,  # 10B tokens
    batch_size_per_gpu=32,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    checkpoint_every=5000,
    eval_every=2000,
    log_every=100,
    eval_samples=1000,
    mixed_precision="bf16",  # Use bf16 on A40
    seed=42,
)

# WandB configuration
wandb_config = WandbConfig(
    project="recursive-reasoning-trm",
    name="pretrain-gpt2-small-10B",
    tags=["pretraining", "gpt2-small", "fineweb-edu", "10B-tokens"],
    notes="Pretraining GPT-2 Small with 64 latent tokens on FineWeb-Edu (10B tokens)",
    mode="online",  # Change to "offline" if no internet
)

# Paths
CHECKPOINT_DIR = project_root / "checkpoints" / "pretrain"
LOG_DIR = project_root / "logs" / "pretrain"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Resume from checkpoint?
RESUME = True  # Set to True to resume from latest checkpoint

print("✓ Configuration loaded")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Log dir: {LOG_DIR}")

## Initialize Accelerator and Hardware Optimization

In [ ]:
# Enable TF32 for A40 speedup
enable_tf32()

# Initialize Accelerator (handles multi-GPU, mixed precision)
accelerator = Accelerator(
    mixed_precision=train_config.mixed_precision,
    gradient_accumulation_steps=train_config.gradient_accumulation_steps,
    log_with="wandb" if wandb_config.mode == "online" else None,
)

print(f"✓ Accelerator initialized")
print(f"  Device: {accelerator.device}")
print(f"  Num processes: {accelerator.num_processes}")
print(f"  Mixed precision: {train_config.mixed_precision}")

## Initialize Model

In [ ]:
# Set random seed
torch.manual_seed(train_config.seed)

# Initialize model
model = HybridTRM(model_config)

# Print model info
if accelerator.is_main_process:
    print_model_info(model)

print("✓ Model initialized")

## Initialize Tokenizer and Dataloaders

In [ ]:
# Initialize tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Training dataloader (streaming)
train_dataloader = get_pretrain_dataloader(
    config=train_config,
    model_config=model_config,
    tokenizer=tokenizer,
    split="train",
)

# Validation dataloader (we'll sample from training set)
print("Creating validation dataset (sampling 1000 examples)...")
eval_examples = create_validation_dataset(
    train_dataloader,
    num_samples=train_config.eval_samples,
)

# Create eval dataloader from fixed examples
eval_dataloader = torch.utils.data.DataLoader(
    eval_examples,
    batch_size=train_config.batch_size_per_gpu,
    shuffle=False,
)

print("✓ Dataloaders initialized")
print(f"  Eval samples: {len(eval_examples)}")

## Initialize Optimizer and Scheduler

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_config.learning_rate,
    weight_decay=train_config.weight_decay,
    betas=(train_config.beta1, train_config.beta2),
    eps=train_config.eps,
)

# Calculate total training steps
batch_size = train_config.batch_size_per_gpu
seq_len = model_config.seq_len_x + model_config.seq_len_y
num_gpus = accelerator.num_processes
tokens_per_step = batch_size * seq_len * num_gpus * train_config.gradient_accumulation_steps
num_training_steps = train_config.target_tokens // tokens_per_step

# Learning rate scheduler
scheduler = setup_scheduler(
    optimizer=optimizer,
    train_config=train_config,
    num_training_steps=num_training_steps,
)

print("✓ Optimizer and scheduler initialized")
print(f"  Total training steps: {num_training_steps:,}")
print(f"  Tokens per step: {tokens_per_step:,}")

## Prepare for Distributed Training

In [ ]:
# Prepare everything with Accelerator (for DDP, mixed precision, etc.)
model, optimizer, train_dataloader, eval_dataloader, scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, scheduler
)

print("✓ Model and dataloaders prepared for distributed training")

## Load Checkpoint (if resuming)

In [ ]:
start_step = 0
start_epoch = 0

if RESUME:
    latest_checkpoint = get_latest_checkpoint(str(CHECKPOINT_DIR))
    if latest_checkpoint:
        print(f"Resuming from checkpoint: {latest_checkpoint}")
        metadata = load_checkpoint(
            path=latest_checkpoint,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            accelerator=accelerator,
        )
        start_step = metadata["step"]
        start_epoch = metadata["epoch"]
    else:
        print("No checkpoint found, starting from scratch")
else:
    print("Starting training from scratch")

## Initialize Weights & Biases

In [ ]:
# Setup WandB logging
setup_wandb(
    config=wandb_config,
    train_config=train_config,
    model_config=model_config,
    accelerator=accelerator,
)

## Training Loop

In [ ]:
# Run training
train(
    model=model,
    train_dataloader=train_dataloader,
    eval_dataloader=eval_dataloader,
    optimizer=optimizer,
    scheduler=scheduler,
    accelerator=accelerator,
    model_config=model_config,
    train_config=train_config,
    checkpoint_dir=str(CHECKPOINT_DIR),
    log_dir=str(LOG_DIR),
    start_step=start_step,
    start_epoch=start_epoch,
)

## Cleanup

In [ ]:
# Finish WandB run
finish_wandb(accelerator)

print("\n✓ Pretraining completed!")
print(f"Final checkpoint: {CHECKPOINT_DIR}/checkpoint_final.pt")

## Quick Test: Generate Sample Text

Test the pretrained model with a simple generation example.

In [ ]:
# Test generation
model.eval()

prompt = "The theory of relativity states that"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(accelerator.device)

with torch.no_grad():
    generated_ids = model.generate(
        x_ids=input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
    )

generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"\nPrompt: {prompt}")
print(f"Generated: {generated_text}")